In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [18]:
import torch
import torch.nn as nn
import numpy as np
from src.physics import calcular_loss 

class MockModel(nn.Module):
    def __init__(self, viscosidad_fija=0.01):
        super().__init__()
        # Definimos la viscosidad fija que queremos testear
        self.visc = nn.Parameter(torch.log(torch.tensor([viscosidad_fija], dtype=torch.float32)))

    def forward(self, x, y):
        # CASO DE TEST 1: Forzamos una solución analítica simple
        # ux = x^2 + y^2
        # uy = x * y
        ux = x**2 + y**2
        uy = x * y
        return ux, uy

    def obtener_viscosidad(self):
        return torch.exp(self.visc)


def test_calcular_loss_analitico():
    print("\n--- Corriendo Test de Pérdida Física ---")
    
    # 1. Definimos puntos de prueba simples (N=1)
    # x = 2.0, y = 3.0
    x = torch.tensor([[2.0]], dtype=torch.float32, requires_grad=True)
    y = torch.tensor([[3.0]], dtype=torch.float32, requires_grad=True)
    
    # 2. Inicializamos nuestro modelo controlado con nu = 0.01
    nu_test = 0.01
    modelo_prueba = MockModel(viscosidad_fija=nu_test)
    
    # 3. Calculamos la loss con tu función
    loss_calculada = calcular_loss(modelo_prueba, x, y)
    
    # 4. CÁLCULO TEÓRICO A MANO (Evaluado en x=2, y=3):
    # ux = x^2 + y^2 = 4 + 9 = 13
    # uy = x*y = 6
    #
    # Derivadas de ux:
    # dux_dx = 2x  (= 4)  | dux_dy = 2y  (= 6)
    # dux_dxx = 2         | dux_dyy = 2
    #
    # Derivadas de uy:
    # duy_dx = y   (= 3)  | duy_dy = x   (= 2)
    # duy_dxx = 0         | duy_dyy = 0
    #
    # Residuo X teórico: ux * dux_dx + uy * dux_dy - nu * (dux_dxx + dux_dyy)
    # res_x = 13 * 4 + 6 * 6 - 0.01 * (2 + 2)
    # res_x = 52 + 36 - 0.04 = 87.96
    #
    # Residuo Y teórico: ux * duy_dx + uy * duy_dy - nu * (duy_dxx + duy_dyy)
    # res_y = 13 * 3 + 6 * 2 - 0.01 * (0 + 0)
    # res_y = 39 + 12 - 0 = 51.0
    #
    # Loss esperada = res_x^2 + res_y^2
    loss_esperada = (87.96)**2 + (51.0)**2
    
    print(f"Loss calculada por tu código: {loss_calculada.item():.4f}")
    print(f"Loss esperada teóricamente:   {loss_esperada:.4f}")
    
    # Test de tolerancia (comparamos que sean prácticamente iguales)
    diferencia = abs(loss_calculada.item() - loss_esperada)
    
    assert diferencia < 1e-3, f"¡Error! La loss difiere por {diferencia:.4f}"
    print("¡Test PASADO exitosamente! Autograd está derivando correctamente Burgers 2D.")

if __name__ == "__main__":
    test_calcular_loss_analitico()


--- Corriendo Test de Pérdida Física ---
Loss calculada por tu código: 10337.9609
Loss esperada teóricamente:   10337.9616
¡Test PASADO exitosamente! Autograd está derivando correctamente Burgers 2D.
